# Imports + important definitions:

In [18]:
#Imports:

from chime.calibration import load_Learmonth_data
from scipy import interpolate

import numpy as np
import matplotlib.pyplot as plt
import chime
from scipy.optimize import curve_fit
import glob
from tqdm import trange
import os
from datetime import datetime
import pandas as pd 

#from chime.util import get_day_number, get_day

# Defining the definitions where Jy is more important here

In [11]:
#Looping process to get the median data for that day:

learmonth = "/home/scratch/dbautist/CHIME_archive/learmonthData/"
learmonth_files = glob.glob(f'{learmonth}/L250506*SRD')

def daily_median(path, frequency):  #def in solar flux units
    
    '''Definition for mapping through each day in the wild card
    file path for the month(for example, January of 2026), maps out 
    and calculates the median for each day in that month in units 
    of Solar Flux Units'''

    df = load_Learmonth_data(path)
    med = np.nanmedian(df[frequency])
   
    return med

def daily_median_Jy(path, frequency):  #def in units of Janskys
    
    '''Same thing as the previous function except med is now being
    mulitplied by 10000 to convert SFU into units of Janksys'''

    df = load_Learmonth_data(path)
    med = np.nanmedian(df[frequency]) * 10000
    return med
                 

# defining lists and appending the data 

In [12]:
median_list = []  # defining empty lists 
median_list_Jy = [] #list of medians for each day in units of Janskys 

for i in trange(len(learmonth_files)):     
    output = daily_median(learmonth_files[i], '410')
    output_Jy = daily_median_Jy(learmonth_files[i], '410')
#appending the files:
    median_list.append(output)    
    median_list_Jy.append(output_Jy)

100%|██████████| 1/1 [00:10<00:00, 10.01s/it]


In [13]:
#Organizing by months + seeing trends:

months = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
month_dict = {}

for month in months:
    string = ""
    month_dict[month] = glob.glob(f'{learmonth}/L25{month}*SRD')

In [14]:
print(months)

['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']


In [15]:
day_path = month_dict['12'][0]  #for a singular month as of right now
df = load_Learmonth_data(day_path)

In [16]:
print(df)

                           time  seconds    245   410   610  1415  2695  4975  \
0     2025-12-01 00:00:00+00:00      0.0  107.0  56.0  63.0  -2.0   3.0  -2.0   
1     2025-12-01 00:00:01+00:00      1.0  118.0  56.0  63.0  -2.0   4.0  -2.0   
2     2025-12-01 00:00:02+00:00      2.0  118.0  55.0  63.0  -1.0   0.0  -2.0   
3     2025-12-01 00:00:03+00:00      3.0    NaN   NaN   NaN   NaN   NaN   NaN   
4     2025-12-01 00:00:04+00:00      4.0    NaN   NaN   NaN   NaN   NaN   NaN   
...                         ...      ...    ...   ...   ...   ...   ...   ...   
86395 2025-12-01 23:59:55+00:00  86395.0   22.0  48.0  61.0  -2.0   0.0  -2.0   
86396 2025-12-01 23:59:56+00:00  86396.0   23.0  49.0  61.0  -2.0   3.0  -2.0   
86397 2025-12-01 23:59:57+00:00  86397.0   23.0  48.0  61.0  -2.0   1.0  -2.0   
86398 2025-12-01 23:59:58+00:00  86398.0   22.0  48.0  61.0  -2.0   2.0  -2.0   
86399 2025-12-01 23:59:59+00:00  86399.0   21.0  48.0  62.0  -1.0   4.0  -3.0   

       8800  15400  
0     

In [17]:
print(['01'])
my_list = ['01']
item = my_list[0]
print(item)

['01']
01


# Filtering process

In [18]:
#Looping process:

def filtering(df):
    median = (not np.isnan(np.nanmedian(df['410']))) and np.nanmedian(df['410']) !=1 and np.nanmedian(df['410'])
    result = median
    return result 

good_data = []
bad_data = []
good_date = []
bad_date = []

for month in months:
    for i in trange(len(month_dict[month])):
        path = month_dict[month][i]
        df = load_Learmonth_data(path)
        
        if filtering(df):
            good_data.append(path)
        else:
            bad_data.append(path)
                

100%|██████████| 31/31 [02:30<00:00,  4.84s/it]


In [19]:
print(good_data)

['/home/scratch/dbautist/CHIME_archive/learmonthData/L250101.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250102.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250103.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250104.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250105.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250106.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250107.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250108.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250109.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250110.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250111.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250112.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250113.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250114.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250115.SRD', '/home/sc

In [20]:
for date in good_data:
    base = os.path.basename(date)[3:7]
    num_date = int(base)
    good_date.append(num_date)
    
for nodate in bad_data:
    nobase = os.path.basename(nodate)[3:7]
    no_num_date = int(nobase)
    bad_date.append(no_num_date)

In [21]:
print(bad_date)

[118, 824, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027]


In [22]:
print(day_path)
print(os.path.basename(day_path))
date_name = os.path.basename(day_path)
datetime.strptime(date_name, "L%y%m%d.SRD")


/home/scratch/dbautist/CHIME_archive/learmonthData/L251201.SRD
L251201.SRD


datetime.datetime(2025, 12, 1, 0, 0)

In [23]:
print(good_data)

['/home/scratch/dbautist/CHIME_archive/learmonthData/L250101.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250102.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250103.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250104.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250105.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250106.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250107.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250108.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250109.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250110.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250111.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250112.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250113.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250114.SRD', '/home/scratch/dbautist/CHIME_archive/learmonthData/L250115.SRD', '/home/sc

In [102]:





bad_list = [datetime.strptime("L250119.SRD", "L%y%m%d.SRD"),
    datetime.strptime("L250120.SRD", "L%y%m%d.SRD"),
    datetime.strptime("L250719.SRD", "L%y%m%d.SRD"),
    datetime.strptime("L250720.SRD", "L%y%m%d.SRD"),
    datetime.strptime("L250830.SRD", "L%y%m%d.SRD"),
    datetime.strptime("L250831.SRD", "L%y%m%d.SRD"),
    datetime.strptime("L250901.SRD", "L%y%m%d.SRD")]
bad_list.sort()
print(bad_list)

#bad_dates = sorted(bad_date + dont_exist)
#print(bad_dates)




[datetime.datetime(2025, 1, 19, 0, 0), datetime.datetime(2025, 1, 20, 0, 0), datetime.datetime(2025, 7, 19, 0, 0), datetime.datetime(2025, 7, 20, 0, 0), datetime.datetime(2025, 8, 30, 0, 0), datetime.datetime(2025, 8, 31, 0, 0), datetime.datetime(2025, 9, 1, 0, 0)]


In [25]:
#good vs bad datetime-the actual lists of the datetimes

good_datetime = []
bad_datetimes = []

for good in good_data:
    file = os.path.basename(good)
    new_var = datetime.strptime(file, "L%y%m%d.SRD")
    good_datetime.append(new_var)
    
print(good_datetime)

for bad in bad_data:
    fileb = os.path.basename(bad)
    bad_var = datetime.strptime(fileb, "L%y%m%d.SRD")
    bad_datetimes.append(bad_var)

    



[datetime.datetime(2025, 1, 1, 0, 0), datetime.datetime(2025, 1, 2, 0, 0), datetime.datetime(2025, 1, 3, 0, 0), datetime.datetime(2025, 1, 4, 0, 0), datetime.datetime(2025, 1, 5, 0, 0), datetime.datetime(2025, 1, 6, 0, 0), datetime.datetime(2025, 1, 7, 0, 0), datetime.datetime(2025, 1, 8, 0, 0), datetime.datetime(2025, 1, 9, 0, 0), datetime.datetime(2025, 1, 10, 0, 0), datetime.datetime(2025, 1, 11, 0, 0), datetime.datetime(2025, 1, 12, 0, 0), datetime.datetime(2025, 1, 13, 0, 0), datetime.datetime(2025, 1, 14, 0, 0), datetime.datetime(2025, 1, 15, 0, 0), datetime.datetime(2025, 1, 16, 0, 0), datetime.datetime(2025, 1, 17, 0, 0), datetime.datetime(2025, 1, 21, 0, 0), datetime.datetime(2025, 1, 22, 0, 0), datetime.datetime(2025, 1, 23, 0, 0), datetime.datetime(2025, 1, 24, 0, 0), datetime.datetime(2025, 1, 25, 0, 0), datetime.datetime(2025, 1, 26, 0, 0), datetime.datetime(2025, 1, 27, 0, 0), datetime.datetime(2025, 1, 28, 0, 0), datetime.datetime(2025, 1, 29, 0, 0), datetime.datetime(20

In [27]:

date_str = os.path.basename(day_path)[1:-4]  #from other code
time_UTC = datetime.strptime(date_str, "%y%m%d")
int(time_UTC.strftime("%j"))

335

In [28]:
print(good_date)

[101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 527, 528, 529, 530, 531, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 612, 613, 614, 615, 616, 617, 618, 619, 620, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 701, 702, 703, 704, 705, 706, 707, 708, 709, 710, 711, 712, 713, 714, 715, 716, 717, 718, 721, 722, 723, 724,

In [29]:
#from github

def get_day_number(filepath):
    date_str = os.path.basename(filepath)[1:-4]
    time_UTC = datetime.strptime(date_str, "%y%m%d")
    return int(time_UTC.strftime("%j"))



In [30]:
#from github

def get_day(path):
    filename = os.path.basename(path).replace(".SRD", "")
    YYMMDD = filename[1:]
    datetime_obj = datetime.strptime(YYMMDD, "%y%m%d")
    return int(datetime_obj.strftime("%j"))



# Only applying the good data to y

In [31]:
flux_410 = []  #flux_410 = data that is good 


for i in trange(len(good_data)):
    p = good_data[i]
    df = load_Learmonth_data(p) 
    flux_410.append(np.nanmedian(df['410']))

flux_Jy = [x * 10000 for x in flux_410] #flux_Jy = flux_410 just in units of Janskys

print(flux_Jy)

100%|██████████| 342/342 [26:58<00:00,  4.73s/it]

[420000.0, 480000.0, 430000.0, 460000.0, 430000.0, 420000.0, 410000.0, 420000.0, 420000.0, 430000.0, 490000.0, 460000.0, 440000.0, 460000.0, 450000.0, 460000.0, 530000.0, 710000.0, 440000.0, 620000.0, 530000.0, 480000.0, 450000.0, 440000.0, 420000.0, 400000.0, 410000.0, 400000.0, 410000.0, 420000.0, 470000.0, 420000.0, 400000.0, 430000.0, 420000.0, 460000.0, 450000.0, 440000.0, 420000.0, 400000.0, 420000.0, 440000.0, 420000.0, 440000.0, 440000.0, 460000.0, 420000.0, 430000.0, 480000.0, 450000.0, 400000.0, 430000.0, 450000.0, 420000.0, 420000.0, 400000.0, 380000.0, 380000.0, 380000.0, 390000.0, 430000.0, 430000.0, 400000.0, 410000.0, 410000.0, 420000.0, 400000.0, 430000.0, 410000.0, 420000.0, 450000.0, 440000.0, 480000.0, 450000.0, 400000.0, 400000.0, 410000.0, 430000.0, 390000.0, 420000.0, 290000.0, 430000.0, 400000.0, 410000.0, 420000.0, 380000.0, 370000.0, 420000.0, 430000.0, 450000.0, 490000.0, 500000.0, 460000.0, 460000.0, 400000.0, 400000.0, 410000.0, 440000.0, 440000.0, 460000.0,

# The plot including Estimated bad data:

In [32]:
dates = sorted(good_date + bad_date)
print(dates)

[101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 527, 528, 529, 530, 531, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 612, 613, 614, 615, 616, 617, 618, 619, 620, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 701, 702, 703, 704, 705, 706, 707, 708, 709, 710, 711, 712, 713, 714, 715, 716, 717, 718, 721, 722, 723,

In [33]:
known_jan_points = list(range(101,132))
print(known_jan_points)

known_feb_points = list(range(201, 229))
known_mar_points = list(range(301, 332))
known_apr_points = list(range(401, 431))
known_may_points = list(range(501, 532))
known_jun_points = list(range(601, 631))
known_jul_points = list(range(701, 732))
known_aug_points = list(range(801, 832))
known_sep_points = list(range(901, 931))
known_oct_points = list(range(1001, 1032))
known_nov_points = list(range(1101, 1131))
known_dec_points = list(range(1201, 1232))

print(known_jul_points)

[101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131]
[701, 702, 703, 704, 705, 706, 707, 708, 709, 710, 711, 712, 713, 714, 715, 716, 717, 718, 719, 720, 721, 722, 723, 724, 725, 726, 727, 728, 729, 730, 731]


In [35]:
#converts the list of ba_dates into datetime objects 

datetime_obj = []
for num in bad_date:
    date_str = f"{int(num):04d}"
    full_date_str = f"2025{date_str}"
    
    dt_obj = datetime.strptime(full_date_str, "%Y%m%d")
    datetime_obj.append(dt_obj)
    
print(datetime_obj)

[datetime.datetime(2025, 1, 18, 0, 0), datetime.datetime(2025, 8, 24, 0, 0), datetime.datetime(2025, 10, 14, 0, 0), datetime.datetime(2025, 10, 15, 0, 0), datetime.datetime(2025, 10, 16, 0, 0), datetime.datetime(2025, 10, 17, 0, 0), datetime.datetime(2025, 10, 18, 0, 0), datetime.datetime(2025, 10, 19, 0, 0), datetime.datetime(2025, 10, 20, 0, 0), datetime.datetime(2025, 10, 21, 0, 0), datetime.datetime(2025, 10, 22, 0, 0), datetime.datetime(2025, 10, 23, 0, 0), datetime.datetime(2025, 10, 24, 0, 0), datetime.datetime(2025, 10, 25, 0, 0), datetime.datetime(2025, 10, 26, 0, 0), datetime.datetime(2025, 10, 27, 0, 0)]


In [40]:
#getting out the bad values:

real1 = dates[0:29]
real2 = dates[179:208]
real3 = dates[207:237]
real4 = dates[237:266]


dont_existJ = [item for item in known_jan_points if item not in real1]
dont_existJ

dont_existJu = [item for item in known_jul_points if item not in real2]
dont_existJu

dont_existaug = [item for item in known_aug_points if item not in real3]
dont_existaug

dont_existsep = [item for item in known_sep_points if item not in real4]
dont_existsep


[901]

In [41]:
#if statements for months with a certian # of days:

#months with 30 days: April, June, September, November
#months with 31 days: January, March, May, July, August, October, December
#months with 28 days: Febuary 

In [42]:
real4 = dates[237:267]
real4

[902,
 903,
 904,
 905,
 906,
 907,
 908,
 909,
 910,
 911,
 912,
 913,
 914,
 915,
 916,
 917,
 918,
 919,
 920,
 921,
 922,
 923,
 924,
 925,
 926,
 927,
 928,
 929,
 930,
 1001]

In [43]:
good_datetime[0:29]

[datetime.datetime(2025, 1, 1, 0, 0),
 datetime.datetime(2025, 1, 2, 0, 0),
 datetime.datetime(2025, 1, 3, 0, 0),
 datetime.datetime(2025, 1, 4, 0, 0),
 datetime.datetime(2025, 1, 5, 0, 0),
 datetime.datetime(2025, 1, 6, 0, 0),
 datetime.datetime(2025, 1, 7, 0, 0),
 datetime.datetime(2025, 1, 8, 0, 0),
 datetime.datetime(2025, 1, 9, 0, 0),
 datetime.datetime(2025, 1, 10, 0, 0),
 datetime.datetime(2025, 1, 11, 0, 0),
 datetime.datetime(2025, 1, 12, 0, 0),
 datetime.datetime(2025, 1, 13, 0, 0),
 datetime.datetime(2025, 1, 14, 0, 0),
 datetime.datetime(2025, 1, 15, 0, 0),
 datetime.datetime(2025, 1, 16, 0, 0),
 datetime.datetime(2025, 1, 17, 0, 0),
 datetime.datetime(2025, 1, 21, 0, 0),
 datetime.datetime(2025, 1, 22, 0, 0),
 datetime.datetime(2025, 1, 23, 0, 0),
 datetime.datetime(2025, 1, 24, 0, 0),
 datetime.datetime(2025, 1, 25, 0, 0),
 datetime.datetime(2025, 1, 26, 0, 0),
 datetime.datetime(2025, 1, 27, 0, 0),
 datetime.datetime(2025, 1, 28, 0, 0),
 datetime.datetime(2025, 1, 29, 0,

In [44]:
#combining the lists into one list:

dont_exist = sorted(dont_existJ + dont_existJu + dont_existaug, reverse=True)
dont_exist

[831, 830, 720, 719, 120, 119]

In [45]:
#adding them to bad date:

bad_dates = sorted(bad_date + dont_exist)
print(bad_dates)

[118, 119, 120, 719, 720, 824, 830, 831, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027]


In [46]:
#converting 'dates' into a datetime object:

b_datetime = []

for num in bad_dates:
    date_str = f"{num:06d}"
    dt_obj = datetime.strptime(date_str, "%y%m%d")
    formatted_str = dt_obj.strftime("L%y%m%d.SRD")
    b_datetime.append(formatted_str)
    
print(b_datetime)

#datetime.strptime(date_name, "L%y%m%d.SRD")\
#for bad in bad_data:
    #fileb = os.path.basename(bad)
    #bad_var = datetime.strptime(fileb, "L%y%m%d.SRD")
    #bad_datetimes.append(bad_var)

['L000118.SRD', 'L000119.SRD', 'L000120.SRD', 'L000719.SRD', 'L000720.SRD', 'L000824.SRD', 'L000830.SRD', 'L000831.SRD', 'L001014.SRD', 'L001015.SRD', 'L001016.SRD', 'L001017.SRD', 'L001018.SRD', 'L001019.SRD', 'L001020.SRD', 'L001021.SRD', 'L001022.SRD', 'L001023.SRD', 'L001024.SRD', 'L001025.SRD', 'L001026.SRD', 'L001027.SRD']


In [47]:
#plot in units of Janskys:

f = interpolate.interp1d((good_date), (flux_Jy)) #interpulating good_date

ynew = f(good_date)
yfilled = f(bad_dates)



In [48]:
print(yfilled)
len(yfilled)

[575000.         620000.         665000.         423333.33333333
 406666.66666667 410000.         509178.08219178 508356.16438356
 427333.33333333 424666.66666667 422000.         419333.33333333
 416666.66666667 414000.         411333.33333333 408666.66666667
 406000.         403333.33333333 400666.66666667 398000.
 395333.33333333 392666.66666667]


22

In [49]:
len(yfilled)

22

# Pandas Dataframe:

In [50]:
#putting it into a pandas file:

good_flux_data = {
    "Flux": flux_Jy,
    "Date": good_datetime,
    "Source": "Real"
}

good = pd.DataFrame(good_flux_data)
good

,Flux,Date,Source
0,420000.0,2025-01-01,Real
1,480000.0,2025-01-02,Real
2,430000.0,2025-01-03,Real
3,460000.0,2025-01-04,Real
4,430000.0,2025-01-05,Real
...,...,...,...
337,400000.0,2025-12-27,Real
338,420000.0,2025-12-28,Real
339,420000.0,2025-12-29,Real
340,360000.0,2025-12-30,Real


In [51]:
#bad data

bad_flux_data = {
    "Flux": yfilled,
    "Date": bad_datetime,
    "Source": "Interpoliated"
}

bad = pd.DataFrame(bad_flux_data)
bad



NameError: name 'bad_datetime' is not defined

In [52]:
frames = [good, bad]
result = pd.concat(frames)
result
result_sort = result.sort_values(by=["Date"])
result_sort




TypeError: cannot concatenate object of type '<class 'str'>'; only Series and DataFrame objs are valid

In [53]:
result_sort["Date"] == str

result_sort["Date"] == str

type(result_sort["Date"][0])



NameError: name 'result_sort' is not defined

In [54]:
result_sort.to_csv("Daily_Calibration.csv", index=False)

NameError: name 'result_sort' is not defined

In [ ]:
print(result_sort)

NameError: name 'result_sort' is not defined

In [ ]:
pd.read_csv("Daily_Calibration.csv")
dataframe = pd.read_csv("Daily_Calibration.csv")


In [ ]:
target_date = dataframe['Date'][300] #gui_reduction

y_target = dataframe.loc[dataframe['Date'] == target_date, 'Flux'].values[0]
print(y_target)


460000.0


In [ ]:
dataframe['Date'] = pd.to_datetime(dataframe['Date'])
dataframe = dataframe.sort_values(by='Date').reset_index(drop=True)
dataframe['Date'] = dataframe['Date'].dt.strftime('%y-%m-%d')

for index in range(len(dataframe)):
    da = dataframe.iloc[index]['Date']
    y_variable = dataframe.loc[dataframe['Date'] == da, 'Flux'].values[0]
    
print(y_variable)

370000.0


In [ ]:
for i in range(len(dataframe)):
    if 12 == dataframe.Date[i]:
        indx = i
        
dataframe.iloc[indx]

NameError: name 'indx' is not defined

In [ ]:
for i in 

,Flux,Date,Source
0,420000.0,25-01-01,Real
1,480000.0,25-01-02,Real
2,430000.0,25-01-03,Real
3,460000.0,25-01-04,Real
4,430000.0,25-01-05,Real
...,...,...,...
353,400000.0,25-12-27,Real
354,420000.0,25-12-28,Real
355,420000.0,25-12-29,Real
356,360000.0,25-12-30,Real


In [ ]:
indx = []
df = []

for i in range(len(dataframe)):
    if "11" == dataframe['Date'][i]:
        indx.append(i)
    
df = pd.DataFrame()

for indexes in indx:
    df = df.append(dataframe.iloc[indexes])
    
df = dataframe.where(dataframe['Flux'] == 150)
df.dropna()
df


,Flux,Date,Source
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN
...,...,...,...
353,NaN,NaN,NaN
354,NaN,NaN,NaN
355,NaN,NaN,NaN
356,NaN,NaN,NaN


In [ ]:
print(dataframe['Date'].iloc[0:389])

0      25-01-01
1      25-01-02
2      25-01-03
3      25-01-04
4      25-01-05
         ...   
353    25-12-27
354    25-12-28
355    25-12-29
356    25-12-30
357    25-12-31
Name: Date, Length: 358, dtype: object


In [ ]:
with pd.option_context('display.max_rows', None):
    print(dataframe.iloc[0:343]['Date'].to_string(index=False))
    
    


25-01-01
25-01-02
25-01-03
25-01-04
25-01-05
25-01-06
25-01-07
25-01-08
25-01-09
25-01-10
25-01-11
25-01-12
25-01-13
25-01-14
25-01-15
25-01-16
25-01-17
25-01-18
25-01-21
25-01-22
25-01-23
25-01-24
25-01-25
25-01-26
25-01-27
25-01-28
25-01-29
25-01-30
25-01-31
25-02-01
25-02-02
25-02-03
25-02-04
25-02-05
25-02-06
25-02-07
25-02-08
25-02-09
25-02-10
25-02-11
25-02-12
25-02-13
25-02-14
25-02-15
25-02-16
25-02-17
25-02-18
25-02-19
25-02-20
25-02-21
25-02-22
25-02-23
25-02-24
25-02-25
25-02-26
25-02-27
25-02-28
25-03-01
25-03-02
25-03-03
25-03-04
25-03-05
25-03-06
25-03-07
25-03-08
25-03-09
25-03-10
25-03-11
25-03-12
25-03-13
25-03-14
25-03-15
25-03-16
25-03-17
25-03-18
25-03-19
25-03-20
25-03-21
25-03-22
25-03-23
25-03-24
25-03-25
25-03-26
25-03-27
25-03-28
25-03-29
25-03-30
25-03-31
25-04-01
25-04-02
25-04-03
25-04-04
25-04-05
25-04-06
25-04-07
25-04-08
25-04-09
25-04-10
25-04-11
25-04-12
25-04-13
25-04-14
25-04-15
25-04-16
25-04-17
25-04-18
25-04-19
25-04-20
25-04-21
25-04-22
25-04-23
2

In [ ]:
dataframe_all = dataframe.iloc[0:359]['Date'].to_string(index=False)
print(dataframe_all)


25-01-01
25-01-02
25-01-03
25-01-04
25-01-05
25-01-06
25-01-07
25-01-08
25-01-09
25-01-10
25-01-11
25-01-12
25-01-13
25-01-14
25-01-15
25-01-16
25-01-17
25-01-18
25-01-21
25-01-22
25-01-23
25-01-24
25-01-25
25-01-26
25-01-27
25-01-28
25-01-29
25-01-30
25-01-31
25-02-01
25-02-02
25-02-03
25-02-04
25-02-05
25-02-06
25-02-07
25-02-08
25-02-09
25-02-10
25-02-11
25-02-12
25-02-13
25-02-14
25-02-15
25-02-16
25-02-17
25-02-18
25-02-19
25-02-20
25-02-21
25-02-22
25-02-23
25-02-24
25-02-25
25-02-26
25-02-27
25-02-28
25-03-01
25-03-02
25-03-03
25-03-04
25-03-05
25-03-06
25-03-07
25-03-08
25-03-09
25-03-10
25-03-11
25-03-12
25-03-13
25-03-14
25-03-15
25-03-16
25-03-17
25-03-18
25-03-19
25-03-20
25-03-21
25-03-22
25-03-23
25-03-24
25-03-25
25-03-26
25-03-27
25-03-28
25-03-29
25-03-30
25-03-31
25-04-01
25-04-02
25-04-03
25-04-04
25-04-05
25-04-06
25-04-07
25-04-08
25-04-09
25-04-10
25-04-11
25-04-12
25-04-13
25-04-14
25-04-15
25-04-16
25-04-17
25-04-18
25-04-19
25-04-20
25-04-21
25-04-22
25-04-23
2

In [ ]:
flux_list = dataframe.iloc[0:359]['Flux'].tolist()
print(flux_list)

In [ ]:
for_reduction = dataframe.iloc[0:359]['Flux']


float_values = for_reduction.astype(float).to_list()
print(float_values)

In [ ]:
df_clean = dataframe.iloc[0:359][['Date', 'Flux']].copy()
df_clean['Flux'] = df_clean['Flux'].astype(float)

print(df_clean['Flux'])

In [ ]:
#loop for dataframe indexing:

for i in 

In [ ]:
print(sr)

In [ ]:
type(sr)

In [ ]:
type(sr["Date"][0:300])

In [ ]:
initial_bound = 0

red = dataframe.iloc[initial_bound]["Date"]

In [ ]:
print(red)

red_flux = dataframe.iloc[initial_bound]["Flux"]
print(red_flux)

In [ ]:
df_clean = dataframe.iloc[0:359][['Date', 'Flux']].copy()
df_clean['Flux'] = df_clean['Flux'].astype(float)

print(df_clean['Flux'])

In [ ]:
clean = dataframe.iloc[0]['Date', 'Flux']

In [ ]:
dataframe['Date'] = pd.to_datetime(dataframe['Date'])
target_date = pd.to_datetime('2025-09-16')
target_date
target_flux = df.loc[dataframe['Date'] == target_date, 'Flux'].values[0]
target_flux

In [ ]:
x = pd.DataFrame([["A", 100, 'D'], ['B', 200, 'E'], ['C', 100, "F"]],
                columns = ["name", "salary", 'department'])

for i in range(len(x)):
    if 200 == x.salary[i]:
        indx = i

x.iloc[indx]

name            B
salary        200
department      E
Name: 1, dtype: object

In [ ]:
dataframe
dataframe['Date']
type(dataframe['Date'])

pandas.core.series.Series

In [ ]:
for i in range(len(dataframe)):
    if 2025_005 == dataframe['Date']:
        indx = i
        
dataframe.iloc[indx]

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:

for i in range(len(dataframe)):
    if 2025_001 == dataframe.Date[i]:
        indx = i
        
target_date = dataframe.iloc[indx]
print(target_date)

Empty DataFrame
Columns: [Flux, Date, Source]
Index: []


In [ ]:
print(dataframe)

         Flux      Date Source
0    420000.0  25-01-01   Real
1    480000.0  25-01-02   Real
2    430000.0  25-01-03   Real
3    460000.0  25-01-04   Real
4    430000.0  25-01-05   Real
..        ...       ...    ...
353  400000.0  25-12-27   Real
354  420000.0  25-12-28   Real
355  420000.0  25-12-29   Real
356  360000.0  25-12-30   Real
357  370000.0  25-12-31   Real

[358 rows x 3 columns]


In [ ]:
indx = []

for i in range(len(dataframe)):
    if 2025_004 == dataframe.Date[i]:
        indx.append[i]
        
df = pd.DataFrame()

for indexes in indx:
    df = df.append(dataframe.iloc[indexes])
    
df = dataframe.where(dataframe.Date == 2025_004)
df.dropna()
df

,Flux,Date,Source
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN
...,...,...,...
353,NaN,NaN,NaN
354,NaN,NaN,NaN
355,NaN,NaN,NaN
356,NaN,NaN,NaN


In [ ]:
indx = []

for i in range(len(dataframe)):
    if 10 == dataframe.Date[i]:
        indx.append(i)
        
dataframe.iloc(indx)

TypeError: unhashable type: 'list'

In [ ]:
dataframe

,Flux,Date,Source
0,420000.0,25-01-01,Real
1,480000.0,25-01-02,Real
2,430000.0,25-01-03,Real
3,460000.0,25-01-04,Real
4,430000.0,25-01-05,Real
...,...,...,...
353,400000.0,25-12-27,Real
354,420000.0,25-12-28,Real
355,420000.0,25-12-29,Real
356,360000.0,25-12-30,Real


In [ ]:
dataframe.Date[10]

'25-01-11'

In [ ]:
divide = [item / 490000 for item in flux_Jy]
divide








NameError: name 'flux_Jy' is not defined

In [6]:
#imports:

from chime.calibration import load_Learmonth_data
from scipy import interpolate

import numpy as np
import glob
from tqdm import trange
import os
from datetime import datetime
import pandas as pd

learmonth = "/home/scratch/dbautist/CHIME_archive/learmonthData/"
learmonth_files = glob.glob(f'{learmonth}/L25*SRD')

#organizing by months + seeing trends:

months = ['10']
month_dict = {}

for month in months:
    string = ""
    month_dict[month] = glob.glob(f'{learmonth}/L25{month}*SRD')
    
day_path = month_dict['10'][0] #for all 12 months 
df = load_Learmonth_data(day_path)

#Filtering Process:

def filtering(df):
    median = (not np.isnan(np.nanmedian(df['410']))) and np.nanmedian(df['410']) !=1 and np.nanmedian(df['410'])
    result = median
    return result 

#empty lists:

good_dat = [] #string of good days of data
bad_dat = [] #string of bad days of data

good_date = []
bad_date = []

#Looping process:

for month in months:
    for i in trange(len(month_dict[month])):
        path = month_dict[month][i]
        df = load_Learmonth_data(path)
        
        if filtering(df):
            good_dat.append(path)
        else:
            bad_dat.append(path)
               
#doing the proper dates:

for date in good_dat:
    base = os.path.basename(date)[3:7]
    num_date = int(base)
    good_date.append(num_date)
    
for nodate in bad_dat:
    nobase = os.path.basename(nodate)[3:7]
    no_num_date = int(nobase)
    bad_date.append(no_num_date)

good_datetime = []
bad_datetimes = []

for good in good_dat:
    file = os.path.basename(good)
    new_var = datetime.strptime(file, "L%y%m%d.SRD")
    good_datetime.append(new_var)
    
for bad in bad_dat:
    fileb = os.path.basename(bad)
    bad_var = datetime.strptime(fileb, "L%y%m%d.SRD")
    bad_datetimes.append(bad_var)
    
bad_datetime = sorted(bad_datetimes + bad_list)
dates = sorted(good_date + bad_date)


100%|██████████| 31/31 [02:35<00:00,  5.03s/it]


NameError: name 'bad_list' is not defined

In [7]:

#calculating the fluxes:

raw_flux = [] #empty list for raw flux values throughout year

for i in trange(len(good_dat)):
    p = good_dat[i]
    df = load_Learmonth_data(p)
    raw_flux.append(np.nanmedian(df['410']))
    
flux_Jy = [x * 10000 for x in raw_flux] #converting from counts to Janskys

#interpolating the data:

datetime_bad = []

known_jan_points = list(range(101, 132))
known_feb_points = list(range(201, 229))
known_mar_points = list(range(301, 332))
known_apr_points = list(range(401, 431))
known_may_points = list(range(501, 532))
known_jun_points = list(range(601, 631))
known_jul_points = list(range(701, 732))
known_aug_points = list(range(801, 832))
known_sept_points = list(range(901, 931))
known_oct_points = list(range(1001, 1131))
known_nov_points = list(range(1101, 1131))
known_dec_points = list(range(1201, 1232))



real1 = dates[0:29]      #Jan
real2 = dates[30:47]
real3 = dates[48:79]
real4 = dates[80:110]
real5 = dates[111:142]
real6 = dates[143:178]
real7 = dates[179:208]   #July
real8 = dates[207:237]   #August
real9 = dates[237:266]   #Sept
real10 = dates[265:296]
real11 = dates[295:325]
real12 = dates[324:265]


dont_existJ = [item for item in known_jan_points if item not in real1]
dont_existJu = [item for item in known_jul_points if item not in real2]
dont_existaug = [item for item in known_aug_points if item not in real3]
dont_existsep = [item for item in known_sept_points if item not in real4]

dont_exist = sorted(dont_existJ + dont_existJu + dont_existaug + dont_existsep)
bad_dates = sorted(bad_date + dont_exist)



100%|██████████| 17/17 [01:24<00:00,  5.00s/it]


NameError: name 'dates' is not defined

In [10]:


dates = sorted(good_date + bad_date)
print(dates)


bad_list = [datetime.strptime("L250119.SRD", "L%y%m%d.SRD"),
    datetime.strptime("L250120.SRD", "L%y%m%d.SRD"),
    datetime.strptime("L250719.SRD", "L%y%m%d.SRD"),
    datetime.strptime("L250720.SRD", "L%y%m%d.SRD"),
    datetime.strptime("L250830.SRD", "L%y%m%d.SRD"),
    datetime.strptime("L250831.SRD", "L%y%m%d.SRD"),
    datetime.strptime("L250901.SRD", "L%y%m%d.SRD")]
bad_list.sort()

bad_datetime = sorted(bad_datetimes + bad_list)
dates = sorted(good_date + bad_date)

print(bad_list)

real1 = dates[0:29]      #Jan
real2 = dates[30:47]
real3 = dates[48:79]
real4 = dates[80:110]
real5 = dates[111:142]
real6 = dates[143:178]
real7 = dates[179:208]   #July
real8 = dates[207:237]   #August
real9 = dates[237:266]   #Sept
real10 = dates[265:296]
real11 = dates[295:325]
real12 = dates[324:265]


dont_existJ = [item for item in known_jan_points if item not in real1]
dont_existJu = [item for item in known_jul_points if item not in real7]
dont_existaug = [item for item in known_aug_points if item not in real8]
dont_existsep = [item for item in known_sept_points if item not in real9]

dont_exist = sorted(dont_existJ + dont_existJu + dont_existaug + dont_existsep)
bad_dates = sorted(bad_date + dont_exist)

print(dont_exist)

[1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030, 1031]
[datetime.datetime(2025, 1, 19, 0, 0), datetime.datetime(2025, 1, 20, 0, 0), datetime.datetime(2025, 7, 19, 0, 0), datetime.datetime(2025, 7, 20, 0, 0), datetime.datetime(2025, 8, 30, 0, 0), datetime.datetime(2025, 8, 31, 0, 0), datetime.datetime(2025, 9, 1, 0, 0)]
[101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 701, 702, 703, 704, 705, 706, 707, 708, 709, 710, 711, 712, 713, 714, 715, 716, 717, 718, 719, 720, 721, 722, 723, 724, 725, 726, 727, 728, 729, 730, 731, 801, 802, 803, 804, 805, 806, 807, 808, 809, 810, 811, 812, 813, 814, 815, 816, 817, 818, 819, 820, 821, 822, 823, 824, 825, 826, 827, 828, 829, 830, 831, 901, 902, 903, 904, 905, 906, 907, 908, 909, 910, 911, 912, 913, 914, 915, 916, 9

In [11]:
len(bad_datetimes)

14

In [12]:

f = interpolate.interp1d((good_date), (flux_Jy))

good_thing = f(good_date)
est_flux = f(bad_date)

datetime_bad = []

for num in bad_date:
    date_str = f"{int(num):04d}"
    full_date_str = f"2025{date_str}"
    
    dt_obj = datetime.strptime(full_date_str, "%Y%m%d")

#Pandas DataFrame:

good_flux_data = {
    "Date": good_datetime,
    "Flux": flux_Jy,
    "Source": "Real"
}

bad_flux_data = {
    "Date": bad_datetimes,
    "Flux": est_flux,
    "Source": "Interpoliated"
}

good = pd.DataFrame(good_flux_data)
bad = pd.DataFrame(bad_flux_data)

total = [good, bad]
full_list = pd.concat(total)
full_list_sort = full_list.sort_values(by=["Date"])

full_list_sort["Date"] == str

full_list_sort.to_csv("filtering_OCT_exist.csv", index=False)


In [13]:
print(bad_datetimes[1])


2025-10-15 00:00:00


In [20]:
plt.figure()
plt.scatter(good_datetime, flux_Jy, label='410MHz')
plt.scatter(bad_datetimes, est_flux, color='red', label='interpolated values')
#m = np.linspace(bad_datetimes[1], bad_datetimes[31])
#plt.plot(m, f(m), color='lightgreen', label='fit')
plt.xticks(rotation=45, ha='right')

plt.legend()
plt.title(f"Median solar flux for October")
plt.ylabel("flux in Janskys")

plt.grid()
#plt.hlines(np.median(flux_410), min(good_data), max(good_data))
plt.legend()

NameError: name 'plt' is not defined

In [2]:
print(good_data)

NameError: name 'good_data' is not defined

In [19]:
print(bad_datetimes)

[datetime.datetime(2025, 10, 14, 0, 0), datetime.datetime(2025, 10, 15, 0, 0), datetime.datetime(2025, 10, 16, 0, 0), datetime.datetime(2025, 10, 17, 0, 0), datetime.datetime(2025, 10, 18, 0, 0), datetime.datetime(2025, 10, 19, 0, 0), datetime.datetime(2025, 10, 20, 0, 0), datetime.datetime(2025, 10, 21, 0, 0), datetime.datetime(2025, 10, 22, 0, 0), datetime.datetime(2025, 10, 23, 0, 0), datetime.datetime(2025, 10, 24, 0, 0), datetime.datetime(2025, 10, 25, 0, 0), datetime.datetime(2025, 10, 26, 0, 0), datetime.datetime(2025, 10, 27, 0, 0)]
